<a href="https://colab.research.google.com/github/arvind47d/florence2train/blob/colab/Fine_tune_Florence_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning Florence-2 on DocVQA

In this notebook, we will fine-tune Florence-2 by MSFT, a new vision language model capable of various tasks, on document question answering.

Note that GH doesn't render rich outputs, so you need to run this tutorial in a Colab T4 notebook to train the model and see the outputs.

Let's start by installing the dependencies and loading the dataset.

In [1]:
!pip install torch=='2.4.1+cu121' torchvision=='0.19.1+cu121' torchaudio=='2.4.1+cu121' --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.9/798.9 MB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 145.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 113.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 111.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 56.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 134.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 21.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 45.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/19

In [2]:
!pip install -q datasets timm einops

In [3]:
!pip install flash_attn --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 128.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for flash_attn: filename=flash_attn-2.8.3-cp312-cp312-linux_x86_64.whl size=255985226 sha256=ee1fbb7dc9d4f6e973687e15e245727d39c2b5f5884c74fa30d21bd1841854af
  Stored in directory: /root/.cache/pip/wheels/3d/59/46/f282c12c73dd4bb3c2e3fe199f1a0d0f8cec06df0cccfeee27
Successfully built flash_attn


In [4]:
import os
import json
import torch
from torch.utils.data import Dataset
from transformers import AutoModelForCausalLM, AutoProcessor, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from PIL import Image

In [5]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Set up paths in Google Drive
DRIVE_BASE_PATH = "/content/drive/MyDrive/Colab_experiments/invoice_training"  # Base path to your Google Drive

# Model and directories configuration
MODEL_ID = "microsoft/Florence-2-base-ft"

# Update these paths to point to your Google Drive folders
OUTPUT_DIR = os.path.join(DRIVE_BASE_PATH, "model_checkpoints_flash")
DATA_DIR = os.path.join(DRIVE_BASE_PATH, "output_trainingdata")

# Alternatively, if you want to specify custom paths:
#OUTPUT_DIR = "/content/drive/MyDrive/Colab_experiments/invoice_training/model_checkpoints_flash"
# DATA_DIR = "/content/drive/MyDrive/your_project_folder/output/output_trainingdata"

ANNOTATIONS_FILE = os.path.join(DATA_DIR, "annotations", "training.jsonl")
IMAGES_DIR = os.path.join(DATA_DIR, "images")

# Check for Flash Attention support
use_flash_attention = True
if not torch.cuda.is_available():
    use_flash_attention = False
    print("CUDA not available, disabling Flash Attention.")
else:
    # Simple check, ideally check capability
    print("CUDA available, attempting to use Flash Attention 2.")

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify paths exist
print(f"Output directory: {OUTPUT_DIR}")
print(f"Data directory: {DATA_DIR}")
print(f"Annotations file: {ANNOTATIONS_FILE}")
print(f"Images directory: {IMAGES_DIR}")

# Check if data directories exist
if os.path.exists(DATA_DIR):
    print(f"✓ Data directory found at {DATA_DIR}")
else:
    print(f"✗ Data directory not found at {DATA_DIR}")
    print("Please ensure your training data is uploaded to Google Drive")

Mounted at /content/drive
CUDA available, attempting to use Flash Attention 2.
Output directory: /content/drive/MyDrive/Colab_experiments/invoice_training/model_checkpoints_flash
Data directory: /content/drive/MyDrive/Colab_experiments/invoice_training/output_trainingdata
Annotations file: /content/drive/MyDrive/Colab_experiments/invoice_training/output_trainingdata/annotations/training.jsonl
Images directory: /content/drive/MyDrive/Colab_experiments/invoice_training/output_trainingdata/images
✓ Data directory found at /content/drive/MyDrive/Colab_experiments/invoice_training/output_trainingdata


In [6]:
class InvoiceDataset(Dataset):
    def __init__(self, jsonl_file, image_dir):
        self.data = []
        self.image_dir = image_dir

        print(f"Loading data from {jsonl_file}...")
        with open(jsonl_file, 'r', encoding='utf-8') as f:
            for line in f:
                self.data.append(json.loads(line))
        print(f"Loaded {len(self.data)} samples.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image_filename = item['image']
        prefix = item['prefix']
        suffix = item['suffix']

        image_path = os.path.join(self.image_dir, image_filename)
        try:
            image = Image.open(image_path).convert("RGB")
        except Exception as e:
            print(f"Error loading image {image_path}: {e}")
            return None

        return {
            "prefix": prefix,
            "suffix": suffix,
            "image": image
        }

In [7]:
# Load Processor
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

# Define Data Collator
def collate_fn(batch):
    # Filter Nones
    batch = [b for b in batch if b is not None]
    if not batch:
        return {}

    images = [item['image'] for item in batch]
    prefixes = [item['prefix'] for item in batch]
    suffixes = [item['suffix'] for item in batch]

    # Construct full text for Causal LM training
    full_texts = [p + s for p, s in zip(prefixes, suffixes)]

    inputs = processor(
        text=full_texts,
        images=images,
        return_tensors="pt",
        padding=True
    )

    # Create labels
    input_ids = inputs["input_ids"]
    labels = input_ids.clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100

    # Ideally we should mask the prompt part, but for simplicity we train on full sequence here.
    # To mask prompt, we would need to tokenize prefix separately and calculate lengths.

    inputs["labels"] = labels

    return inputs

preprocessor_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

processing_florence2.py: 0.00B [00:00, ?B/s]

<unknown>:515: SyntaxWarning: invalid escape sequence '\d'
A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-base-ft:
- processing_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/root/.cache/huggingface/modules/transformers_modules/microsoft/Florence_hyphen_2_hyphen_base_hyphen_ft/f6c1a25888ffc1d945ee8a1a77ac833c7303d46e/processing_florence2.py:515: SyntaxWarning: invalid escape sequence '\d'
  PATTERN: 'r<time_(\d+)><time_(\d+)>([a-zA-Z0-9 ]+)'


tokenizer_config.json:   0%|          | 0.00/34.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

configuration_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-base-ft:
- configuration_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


In [8]:
# Load Model with Flash Attention
model_kwargs = {
    "trust_remote_code": True,
    "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32,
}

if use_flash_attention:
    model_kwargs["attn_implementation"] = "flash_attention_2"

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)

`torch_dtype` is deprecated! Use `dtype` instead!


modeling_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-base-ft:
- modeling_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/463M [00:00<?, ?B/s]

ValueError: Florence2ForConditionalGeneration does not support Flash Attention 2.0 yet. Please request to add support where the model is hosted, on its model hub page: https://huggingface.co/microsoft/Florence-2-base-ft/discussions/new or in the Transformers GitHub repo: https://github.com/huggingface/transformers/issues/new